In [ ]:
import os
import shutil
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

from efficientkan import KAN
from lmkan import LMKAN

# --- 1. Physics & Problem Setup ---
z0, sigma = 0.5, 0.07
eps0 = 0.8 + 0.08j

def get_eps_and_deps(z):
    gaussian = np.exp(-(z - z0)**2 / (2 * sigma**2))
    eps = 1 + (eps0 - 1) * gaussian
    deps = (eps0 - 1) * gaussian * (-(z - z0) / sigma**2)
    return eps, deps

def get_k_and_dk(z, lam):
    k0 = 2 * np.pi / lam
    eps, deps = get_eps_and_deps(z)
    k = k0 * np.sqrt(eps)
    dk = k0 * deps / (2 * np.sqrt(eps))
    return k, dk



def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def train_model(model, name, epochs=2500, lr=1e-3, lam_range=[1/30, 1/10]):
    print(f"\n--- Training Parameterized {name} ---")
    print(f"Number of parameters: {count_parameters(model)}")
    print(f"Wavelength training bounds: [ {lam_range[0]:.4f}, {lam_range[1]:.4f} ]")
    
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    start_time = time.time()
    losses = []
    
    # Random sampling across 2D Parameter Space (z and lambda)
    n_pts = 1000
    for epoch in range(epochs + 1):
        optimizer.zero_grad()
        
        # Resample collocation grids occasionally to prevent memorization
        if epoch % 100 == 0:
            z_train = torch.rand(n_pts, 1, requires_grad=True)
            lam_train = torch.rand(n_pts, 1) * (lam_range[1] - lam_range[0]) + lam_range[0]
            inputs = torch.cat([z_train, lam_train], dim=1)
        
        nn_out = model(inputs)
        
        r_real = torch.atan(z_train) * nn_out[:, 0:1]
        r_imag = torch.atan(z_train) * nn_out[:, 1:2]
        r = torch.complex(r_real, r_imag)
        
        dr_dz = torch.autograd.grad(r, z_train, grad_outputs=torch.ones_like(r), create_graph=True)[0]
        
        k_v, dk_v = get_k_and_dk(z_train.detach().numpy(), lam_train.detach().numpy())
        k_v = torch.from_numpy(k_v).to(torch.complex64)
        dk_v = torch.from_numpy(dk_v).to(torch.complex64)
        
        residual = dr_dz - (dk_v / (2 * k_v)) * (1 - r**2) + 2j * k_v * r
        loss = torch.mean(torch.abs(residual)**2)
        
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        if epoch % 500 == 0:
            print(f"Epoch {epoch} | Gen_Loss: {loss.item():.2e}")
            
    end_time = time.time()
    train_time = end_time - start_time
    print(f"{name} Parametric Training Time: {train_time:.2f} seconds")
    return model, train_time, losses

def predict(model, lam):
    model.eval()
    with torch.no_grad():
        z_train = torch.linspace(0, 1, 2000).view(-1, 1)
        lam_train = torch.full_like(z_train, lam)
        inputs = torch.cat([z_train, lam_train], dim=1)
        
        nn_out = model(inputs)
        r_pred = (torch.atan(z_train) * nn_out[:, 0:2])
        r_pred = torch.complex(r_pred[:, 0], r_pred[:, 1]).numpy()
        
        k_arr, dk_arr = get_k_and_dk(z_train.detach().numpy().flatten(), lam)
        dt_dz = 1j * k_arr - (dk_arr / (2 * k_arr)) * r_pred
        t_pred = np.cumsum(dt_dz) * (1.0/2000)
        u_pred = (np.exp(t_pred) * (1 + r_pred)) / np.sqrt(k_arr)
        u_pred /= u_pred[0]
    return z_train.numpy().flatten(), u_pred

def main():
    
    effkan_model = KAN([2, 12,12, 2], grid_size=5, spline_order=3)
    effkan_model, effkan_time, effkan_losses = train_model(effkan_model, "EfficientKAN", epochs=6000)


    
    lmkan_model = LMKAN(
    in_dim=2,          # (z, lambda)
    out_dim=2,         # (Real, Imag)
    width=4,          # Optimized width
    depth=2,           # Best for physical residual stability
    K=16,              # High-frequency resolution
    basis="fourier_basis",   # Superior for oscillating solutions
    gamma=0.5,         
    metric_hidden=18   # Metric capacity
)
    lmkan_model, lmkan_time, lmkan_losses = train_model(lmkan_model, "LMKAN", epochs=6000)

    # =================================================================
    # Clean output workspace
    # =================================================================
    out_dir = "outputs"
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)

    plt.figure(figsize=(8, 6))
    plt.plot(effkan_losses, '-', color='#D62828', alpha=0.85, lw=2, label=f'PIKAN')
    plt.plot(lmkan_losses, '-', color='#003049', alpha=0.85, lw=2, label=f'LMKAN')
    plt.yscale('log')
    plt.title('Training Loss Convergence (Generalized Over $\lambda$)')
    plt.xlabel('Epochs')
    plt.ylabel('MSE Generalization Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(f"{out_dir}/01_training_loss.pdf", bbox_inches='tight')
    plt.close()

    # =================================================================
    # Generalized Testing Over Discontinuous Lambda Sequences
    # =================================================================
    test_lambdas = [1/15, 1/20, 1/25]

    for lam in test_lambdas:
        lam_label = f"lam_1_{int(np.round(1/lam))}"
        print(f"| Rendering physical plots for Wavelength: $\lambda={lam:.4f}$ -> [ {lam_label} ]")
        
        z_effkan, u_effkan = predict(effkan_model, lam)
        z_lmkan, u_lmkan = predict(lmkan_model, lam)
        
        # --- Reference Solution (TMM baseline) ---
        def helmholtz_sys(z, y):
            u, p = y
            k, _ = get_k_and_dk(z, lam)
            return [p, -k**2 * u]

        k_out, _ = get_k_and_dk(1.0, lam)
        sol = solve_ivp(helmholtz_sys, [1.0, 0.0], [1.0, 1j*k_out], t_eval=np.linspace(1.0, 0.0, 2000))
        z_ref, u_ref = sol.t[::-1], sol.y[0][::-1]
        u_ref /= u_ref[0]

        u_ref_interp = np.interp(z_effkan, z_ref, u_ref)
        err_effkan = np.abs(u_effkan - u_ref_interp)
        err_lmkan = np.abs(u_lmkan - u_ref_interp)

        # ---------------------------------------------------------
        # Plot 2: 1D Wavefield Components Tracker
        # ---------------------------------------------------------
        fig, axs = plt.subplots(2, 2, figsize=(14, 10))
        fig.suptitle(f'Wavefield Components at Generalised Wavelength $\lambda=1/{int(np.round(1/lam))}$', fontsize=16)

        # Re[u]
        axs[0, 0].plot(z_ref, u_ref.real, '-', color='#ADB5BD', lw=5, alpha=0.6, label='Numerical')
        axs[0, 0].plot(z_effkan, u_effkan.real, '--', color='#D62828', lw=2, label='PIKAN')
        axs[0, 0].plot(z_lmkan, u_lmkan.real, ':', color='#003049', lw=2.5, label='LMKAN')
        axs[0, 0].set_title('Real Part: Re[u]')
        axs[0, 0].set_xlabel('z'); axs[0, 0].set_ylabel('Re[u]')
        axs[0, 0].legend(); axs[0, 0].grid(True, alpha=0.3)

        # Im[u]
        axs[0, 1].plot(z_ref, u_ref.imag, '-', color='#ADB5BD', lw=5, alpha=0.6, label='Numerical')
        axs[0, 1].plot(z_effkan, u_effkan.imag, '--', color='#D62828', lw=2, label='PIKAN')
        axs[0, 1].plot(z_lmkan, u_lmkan.imag, ':', color='#003049', lw=2.5, label='LMKAN')
        axs[0, 1].set_title('Imaginary Part: Im[u]')
        axs[0, 1].set_xlabel('z'); axs[0, 1].set_ylabel('Im[u]')
        axs[0, 1].legend(); axs[0, 1].grid(True, alpha=0.3)

        # |u|
        axs[1, 0].plot(z_ref, np.abs(u_ref), '-', color='#ADB5BD', lw=5, alpha=0.6, label='Numerical')
        axs[1, 0].plot(z_effkan, np.abs(u_effkan), '--', color='#D62828', lw=2, label='PIKAN')
        axs[1, 0].plot(z_lmkan, np.abs(u_lmkan), ':', color='#003049', lw=2.5, label='LMKAN')
        axs[1, 0].set_title('Magnitude: |u|')
        axs[1, 0].set_xlabel('z'); axs[1, 0].set_ylabel('|u|')
        axs[1, 0].legend(); axs[1, 0].grid(True, alpha=0.3)

        # Error
        axs[1, 1].plot(z_effkan, err_effkan, '--', color='#D62828', lw=2, label='PIKAN Error')
        axs[1, 1].plot(z_lmkan, err_lmkan, ':', color='#003049', lw=2.5, label='LMKAN Error')
        axs[1, 1].set_title('Absolute Polynomial Error')
        axs[1, 1].set_xlabel('z'); axs[1, 1].set_ylabel('Error Value')
        axs[1, 1].legend(); axs[1, 1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f"{out_dir}/02_{lam_label}_wavefield_1d.pdf", bbox_inches='tight')
        plt.close()

        # ---------------------------------------------------------
        # Plot 3: 2D Spatio-Temporal Propagation
        # ---------------------------------------------------------
        t_vals = np.linspace(0, 1, 150)
        Z_ref, T_ref = np.meshgrid(z_ref, t_vals)
        Z_effkan, T_effkan = np.meshgrid(z_effkan, t_vals)
        Z_lmkan, T_lmkan = np.meshgrid(z_lmkan, t_vals)
        
        U_ref = np.real(u_ref[None, :] * np.exp(-1j * 2 * np.pi * T_ref))
        U_effkan = np.real(u_effkan[None, :] * np.exp(-1j * 2 * np.pi * T_effkan))
        U_lmkan = np.real(u_lmkan[None, :] * np.exp(-1j * 2 * np.pi * T_lmkan))
        
        fig, axs = plt.subplots(1, 3, figsize=(18, 5))
        fig.suptitle(f'Dynamic 2D Wave Propagation $(\lambda=1/{int(np.round(1/lam))})$', fontsize=16)

        vmin = min(U_ref.min(), U_effkan.min(), U_lmkan.min())
        vmax = max(U_ref.max(), U_effkan.max(), U_lmkan.max())

        c1 = axs[0].pcolormesh(Z_ref, T_ref, U_ref, shading='auto', cmap='viridis', vmin=vmin, vmax=vmax)
        axs[0].set_title('Reference Truth')
        axs[0].set_xlabel('Space (z)'); axs[0].set_ylabel('Time (t)')
        fig.colorbar(c1, ax=axs[0])

        c2 = axs[1].pcolormesh(Z_effkan, T_effkan, U_effkan, shading='auto', cmap='viridis', vmin=vmin, vmax=vmax)
        axs[1].set_title('PIKAN Render')
        axs[1].set_xlabel('Space (z)'); axs[1].set_ylabel('Time (t)')
        fig.colorbar(c2, ax=axs[1])

        c3 = axs[2].pcolormesh(Z_lmkan, T_lmkan, U_lmkan, shading='auto', cmap='viridis', vmin=vmin, vmax=vmax)
        axs[2].set_title('LMKAN Render')
        axs[2].set_xlabel('Space (z)'); axs[2].set_ylabel('Time (t)')
        fig.colorbar(c3, ax=axs[2])

        plt.tight_layout()
        plt.savefig(f"{out_dir}/03_{lam_label}_spatiotemporal_2d.pdf", bbox_inches='tight')
        plt.close()
        
        # ---------------------------------------------------------
        # Plot 4: Scatter/Phase Space Portrait
        # ---------------------------------------------------------
        fig, axs = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f'Phase Space Traces Under Wavelength Tension $(\lambda \simeq 1/{int(np.round(1/lam))})$', fontsize=16)

        axs[0].plot(u_ref.real, u_ref.imag, c='gray', lw=0.5, zorder=0, alpha=0.5)
        sc1 = axs[0].scatter(u_ref.real, u_ref.imag, c=z_ref, cmap='plasma', s=8, zorder=1)
        axs[0].set_title('Numerical')
        axs[0].set_xlabel('Re[u]'); axs[0].set_ylabel('Im[u]'); axs[0].set_aspect('equal')
        axs[0].grid(True, alpha=0.3); fig.colorbar(sc1, ax=axs[0], label='z')

        axs[1].plot(u_effkan.real, u_effkan.imag, c='gray', lw=0.5, zorder=0, alpha=0.5)
        sc3 = axs[1].scatter(u_effkan.real, u_effkan.imag, c=z_effkan, cmap='plasma', s=8, zorder=1)
        axs[1].set_title('PIKAN Orbit')
        axs[1].set_xlabel('Re[u]'); axs[1].set_ylabel('Im[u]'); axs[1].set_aspect('equal')
        axs[1].grid(True, alpha=0.3); fig.colorbar(sc3, ax=axs[1], label='z')

        axs[2].plot(u_lmkan.real, u_lmkan.imag, c='gray', lw=0.5, zorder=0, alpha=0.5)
        sc4 = axs[2].scatter(u_lmkan.real, u_lmkan.imag, c=z_lmkan, cmap='plasma', s=8, zorder=1)
        axs[2].set_title('LMKAN Orbit')
        axs[2].set_xlabel('Re[u]'); axs[2].set_ylabel('Im[u]'); axs[2].set_aspect('equal')
        axs[2].grid(True, alpha=0.3); fig.colorbar(sc4, ax=axs[2], label='z')
        
        plt.tight_layout()
        plt.savefig(f"{out_dir}/04_{lam_label}_phasespace_2d.pdf", bbox_inches='tight')
        plt.close()

    print(f"\n[SUCCESS] Custom Head-to-Head Multi-Lambda parameterization executed across all boundaries!")

if __name__ == "__main__":
    main()



--- Training Parameterized EfficientKAN ---
Number of parameters: 1920
Wavelength training bounds: [ 0.0333, 0.1000 ]
Epoch 0 | Gen_Loss: 5.95e+01
Epoch 500 | Gen_Loss: 4.68e-02
Epoch 1000 | Gen_Loss: 4.12e-02
Epoch 1500 | Gen_Loss: 3.14e-02
Epoch 2000 | Gen_Loss: 3.07e-02
Epoch 2500 | Gen_Loss: 2.80e-02
Epoch 3000 | Gen_Loss: 2.40e-02
Epoch 3500 | Gen_Loss: 1.61e-02
Epoch 4000 | Gen_Loss: 9.69e-03
Epoch 4500 | Gen_Loss: 9.95e-03
Epoch 5000 | Gen_Loss: 6.89e-03
Epoch 5500 | Gen_Loss: 4.49e-03
Epoch 6000 | Gen_Loss: 8.79e-03
EfficientKAN Parametric Training Time: 166.42 seconds

--- Training Parameterized LMKAN ---
Number of parameters: 1928
Wavelength training bounds: [ 0.0333, 0.1000 ]
Epoch 0 | Gen_Loss: 2.73e+03
Epoch 500 | Gen_Loss: 1.48e+00
Epoch 1000 | Gen_Loss: 4.76e-01
Epoch 1500 | Gen_Loss: 1.87e-01
Epoch 2000 | Gen_Loss: 6.45e-02
Epoch 2500 | Gen_Loss: 3.19e-02
Epoch 3000 | Gen_Loss: 1.93e-02
Epoch 3500 | Gen_Loss: 1.40e-02
Epoch 4000 | Gen_Loss: 1.18e-02
Epoch 4500 | Gen_Lo